In [70]:
import pandas as pd
import numpy as np

# Data Preprocessing 

In [71]:
mvp_df = pd.read_csv("data/nba_mvp_voting_2010_to_2025.csv")
standings_df = pd.read_csv("data/nba_standings_2010_to_2025.csv")

In [72]:
# Remove Columns not needed for analysis and rename
mvp_df = mvp_df.drop(columns=["Player", "Rank", "Age", "First", "Pts Won", "Pts Max", "WS/48"], errors='ignore')
mvp_df = mvp_df.rename(columns={"Tm": "Team"})
standings_df = standings_df.drop(columns=["W", "L", "PS/G", "PA/G", "SRS", "GB", "Conference"], errors='ignore')


In [73]:
# Map old team abbreviations to new ones
team_abbrev = {
    "NOH": "NOP",
    "CHA": "CHO",
}

mvp_df["Team"] = mvp_df["Team"].replace(team_abbrev)
mvp_df


,Season,Team,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS
0,2009-10,CLE,0.980,76,39.0,29.7,7.3,8.6,1.6,1.0,0.503,0.333,0.767,18.5
1,2009-10,OKC,0.495,82,39.5,30.1,7.6,2.8,1.4,1.0,0.476,0.365,0.900,16.1
2,2009-10,LAL,0.487,73,38.8,27.0,5.4,5.0,1.5,0.3,0.456,0.329,0.811,9.4
3,2009-10,ORL,0.389,82,34.7,18.3,13.2,1.8,0.9,2.8,0.612,0.000,0.592,13.2
4,2009-10,MIA,0.097,77,36.3,26.6,4.8,6.5,1.8,1.1,0.476,0.300,0.761,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-25,MIN,0.012,79,36.3,27.6,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4
203,2024-25,GSW,0.002,70,32.2,24.5,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9
204,2024-25,NYK,0.001,65,35.4,26.0,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3
205,2024-25,LAC,0.001,79,35.3,22.8,5.8,8.7,1.5,0.7,0.410,0.352,0.874,8.3


In [74]:
# Convert games to % of Games Played - to account for shortened seasons and enable the model to generalize better

def get_season_games(season):
    year = int(season.split("-")[1])

    if year == 12:
        return 66
    elif year == 20:
        return 67
    elif year == 21:
        return 72
    else:
        return 82

mvp_df["G"] = mvp_df["G"] / mvp_df["Season"].apply(get_season_games)

In [75]:
# Map team full name to abbreviations in standings_df
team_full_to_abbrev = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "New Jersey Nets": "BRK",
    "Charlotte Hornets": "CHO",
    "Charlotte Bobcats": "CHO",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New Orleans Hornets": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

standings_df['Team'] = standings_df['Team'].map(team_full_to_abbrev)
standings_df

,Season,Team,W/L%
0,2009-10,BOS,0.610
1,2009-10,TOR,0.488
2,2009-10,NYK,0.354
3,2009-10,PHI,0.329
4,2009-10,BRK,0.146
...,...,...,...
475,2024-25,PHO,0.439
476,2024-25,POR,0.439
477,2024-25,SAS,0.415
478,2024-25,NOP,0.256


In [76]:
# Add W/L% to MVP data and drop season and team data now
df = pd.merge(mvp_df, standings_df, on=["Season", "Team"], how='left')
df = df.drop(columns=["Season", "Team"], errors='ignore')
df

,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,W/L%
0,0.980,0.926829,39.0,29.7,7.3,8.6,1.6,1.0,0.503,0.333,0.767,18.5,0.744
1,0.495,1.000000,39.5,30.1,7.6,2.8,1.4,1.0,0.476,0.365,0.900,16.1,0.610
2,0.487,0.890244,38.8,27.0,5.4,5.0,1.5,0.3,0.456,0.329,0.811,9.4,0.695
3,0.389,1.000000,34.7,18.3,13.2,1.8,0.9,2.8,0.612,0.000,0.592,13.2,0.720
4,0.097,0.939024,36.3,26.6,4.8,6.5,1.8,1.1,0.476,0.300,0.761,13.0,0.573
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,0.012,0.963415,36.3,27.6,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4,0.598
203,0.002,0.853659,32.2,24.5,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9,0.585
204,0.001,0.792683,35.4,26.0,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3,0.622
205,0.001,0.963415,35.3,22.8,5.8,8.7,1.5,0.7,0.410,0.352,0.874,8.3,0.610


In [77]:
# Create X and Y datasets and convert them to numpy arrays for model training
X = df.drop(columns=["Share"]).fillna(0).to_numpy()
Y = df[["Share"]].to_numpy().ravel()

In [82]:
# Preprocess test data (2025 - 2026)
test_df = pd.read_csv("data/nba_per_game_stats_2025_to_2026.csv")
test_df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,1.0,Luka Dončić,26.0,LAL,PG,13.0,13.0,37.2,10.8,23.1,...,0.8,7.9,8.8,9.2,1.9,0.5,4.2,2.7,35.2,NaN
1,2.0,Shai Gilgeous-Alexander,27.0,OKC,PG,18.0,18.0,33.1,10.8,19.9,...,0.6,4.3,4.9,6.6,1.5,0.8,1.8,1.8,32.2,NaN
2,3.0,Tyrese Maxey,25.0,PHI,PG,17.0,17.0,39.9,10.8,22.9,...,0.3,4.1,4.4,7.5,1.6,0.8,2.7,2.4,32.2,NaN
3,4.0,Giannis Antetokounmpo,31.0,MIL,PF,13.0,13.0,31.8,12.0,19.1,...,3.7,7.2,10.8,6.8,0.9,1.2,3.5,2.6,31.2,NaN
4,5.0,Donovan Mitchell,29.0,CLE,SG,17.0,17.0,34.0,10.2,20.4,...,0.8,3.9,4.8,5.5,1.4,0.4,3.2,2.2,29.9,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,478.0,Trey Jemison,26.0,NYK,C,2.0,0.0,5.0,0.0,0.0,...,0.5,0.5,1.0,0.0,0.0,0.0,0.5,1.0,0.0,NaN
480,479.0,Christian Koloko,25.0,LAL,C,2.0,0.0,3.0,0.0,0.0,...,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.0,NaN
481,480.0,Chris Mañon,24.0,LAL,SG,2.0,0.0,3.5,0.0,0.5,...,0.0,1.0,1.0,0.0,0.0,0.5,0.0,1.0,0.0,NaN
482,481.0,Garrison Mathews,29.0,IND,SG,2.0,0.0,8.0,0.0,1.5,...,0.0,1.0,1.0,0.5,0.0,0.0,0.5,0.5,0.0,NaN


In [ ]:
# Drop unnecessary columns and rename


# Model Training

In [78]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error 

In [79]:
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

In [80]:
model.fit(X, Y)

,steps,"[('standardscaler', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
